In [ ]:
# default_exp paper.plsr.train_eval

# 3.2. Train & evaluate (PLSR)

> Train & evaluate on multiple train/test splits with different random seeds

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import log_transform_y

from mirzai.training.plsr import (compute_valid_curve, PLS_model, Evaluator)

from fastcore.transform import compose

# Python utilities
#from collections import OrderedDict
#from tqdm.auto import tqdm
#from pathlib import Path

# Data science stack
#import numpy as np
#from sklearn.model_selection import train_test_split
#from sklearn.pipeline import Pipeline
#from sklearn.cross_decomposition import PLSRegression
#from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Wavenumbers:\n {X_names}')
print(f'depth_order (first 3 rows):\n {depth_order[:3, :]}')
print(f'Taxonomic order lookup:\n {tax_lookup}')

X shape: (40132, 1764)
y shape: (40132,)
Wavenumbers:
 [3999 3997 3995 ...  603  601  599]
depth_order (first 3 rows):
 [[43.  2.]
 [ 0.  0.]
 [ 0.  1.]]
Taxonomic order lookup:
 {'alfisols': 0, 'mollisols': 1, 'inceptisols': 2, 'entisols': 3, 'spodosols': 4, 'undefined': 5, 'ultisols': 6, 'andisols': 7, 'histosols': 8, 'oxisols': 9, 'vertisols': 10, 'aridisols': 11, 'gelisols': 12}


## Experiment

### Train on all Soil Taxonomic Orders and test on all and by orders

In [ ]:
# Train on all, test on all, Mollisols, Gelisols and Vertisols
evaluator = Evaluator((X, y), depth_order, seeds=range(2), n_components=40)
evaluator.train_multiple()
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On all test set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=-1))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=-1), '\n')

print('On all test (Mollisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=1))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=1), '\n')

print('On all test (Gelisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=12))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=12), '\n')

print('On all test (Vertisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=10))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=10), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

NameError: name 'X_names' is not defined